# Query-Adaptive Video RAG Pipeline (batch mode, JSON-driven)

Point this at a JSON file of `(video_id, query)` pairs and a Google Drive folder of videos
named by id (`<video_id>.mp4`, `.mov`, etc). For every entry, the matching video is located,
the question is classified into **temporal / spatial / spatio_temporal / counting / audio**,
and only that branch's models run for it — an audio question never loads a vision model, a
counting question never loads Whisper. Per-video results are cached in ChromaDB, so if the
same video shows up across multiple JSON entries its expensive frame/audio processing only
runs once per branch.

**Before running:** Runtime -> Change runtime type -> GPU (T4 or better). You'll be prompted for
an OpenAI API key (GPT-4o-mini for query classification, answer generation, and captioning on
the temporal/spatial branches).

**Evaluation additions in this version:** the pipeline now measures, per question and per category, end-to-end latency, wall-clock time saved vs. a run-every-branch baseline, peak GPU/CPU memory, and how many models actually got loaded (section 17). It also scores the query classifier against human-labelled categories with a confusion matrix, and quantifies how often a misroute produces a wrong final answer (sections 6b, 17c, 17d).


## 1. Install dependencies

In [ ]:
# numpy is pinned in the SAME install command as everything else, so pip resolves one
# consistent version up front instead of needing a force-reinstall later (which caused
# "cannot load module more than once per process" when the runtime didn't fully restart).
!pip install -q "numpy<2.2" scenedetect[opencv] transformers accelerate sentence-transformers \
    chromadb rank_bm25 openai scikit-learn opencv-python-headless soundfile
!apt-get -qq update && apt-get -qq install -y ffmpeg


**The cell above force-kills the kernel on purpose.** Colab will show "session crashed" for a
second and then reconnect automatically — that's the restart taking effect, not a real crash.

Once it reconnects, go to **Runtime -> Run all** (or just run every cell again from the top,
including this install cell — pip will report everything already satisfied and skip straight to
mounting Drive). You should not see the numpy import error again after this point.

## 2. Upload your video

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# --- EDIT THESE THREE PATHS ---
JSON_PATH = "/content/drive/MyDrive/Smoke Test/Smoke Test/MovieChat/query.json"   # your queries+ids JSON
VIDEOS_DIR = "/content/drive/MyDrive/Smoke Test/Smoke Test/MovieChat/Videos"        # folder of videos named by id
OUTPUT_JSON = "/content/drive/MyDrive/path/to/answers.json" # where results are written
# --------------------------------

WORKDIR = "/content/work"
os.makedirs(WORKDIR, exist_ok=True)

assert os.path.isfile(JSON_PATH), f"JSON_PATH not found: {JSON_PATH}"
assert os.path.isdir(VIDEOS_DIR), f"VIDEOS_DIR not found: {VIDEOS_DIR}"
print("JSON_PATH:", JSON_PATH)
print("VIDEOS_DIR:", VIDEOS_DIR)
print("OUTPUT_JSON:", OUTPUT_JSON)


## 3. Load (video_id, query) pairs from JSON, locate each video in Drive

Handles a few common JSON shapes automatically:

- a **list** of objects, each with a video-id-ish key (`video_id`/`id`/`vid`/`video`/`video_name`)
  and a query-ish key (`query`/`question`/`prompt`), optionally with an answer-ish key
  (`answer`/`gt`/`ground_truth`/`gt_answer`) for later comparison against the model's output
- a **dict** keyed by video id, where each value is either a list of query strings, a list of
  query objects (as above), or a single query string

If your JSON uses different key names than these, tell me and I'll adjust `_QUERY_KEYS` /
`_ID_KEYS` / `_ANSWER_KEYS` below.

In [ ]:
import json
import glob

_ID_KEYS = ["video_id", "id", "vid", "video", "video_name", "video_file"]
_QUERY_KEYS = ["query", "question_content", "prompt", "q"]
_ANSWER_KEYS = ["answer", "gt", "ground_truth", "gt_answer", "expected_answer"]
_CATEGORY_KEYS = ["category", "true_category", "gt_category", "label", "true_label"]

def _first_key(d, keys):
    for k in keys:
        if k in d and d[k] not in (None, ""):
            return d[k]
    return None

def load_video_queries(json_path):
    """Returns dict: video_id -> list of {"query", "expected_answer", "true_category" (or None)}"""
    with open(json_path) as f:
        data = json.load(f)

    out = {}

    def add(vid, query, answer=None, category=None):
        vid = str(vid).strip()
        query = str(query).strip()
        if not vid or not query:
            return
        category = str(category).strip().lower() if category else None
        out.setdefault(vid, []).append({"query": query, "expected_answer": answer, "true_category": category})

    if isinstance(data, list):
        for item in data:
            if not isinstance(item, dict):
                continue
            vid = _first_key(item, _ID_KEYS)
            query = _first_key(item, _QUERY_KEYS)
            answer = _first_key(item, _ANSWER_KEYS)
            category = _first_key(item, _CATEGORY_KEYS)
            if vid is None or query is None:
                print(f"  skipping entry, missing id/query keys: {item}")
                continue
            add(vid, query, answer, category)

    elif isinstance(data, dict):
        for vid, val in data.items():
            if isinstance(val, str):
                add(vid, val)
            elif isinstance(val, list):
                for item in val:
                    if isinstance(item, str):
                        add(vid, item)
                    elif isinstance(item, dict):
                        query = _first_key(item, _QUERY_KEYS)
                        answer = _first_key(item, _ANSWER_KEYS)
                        category = _first_key(item, _CATEGORY_KEYS)
                        if query is None:
                            print(f"  skipping entry for {vid}, no query key: {item}")
                            continue
                        add(vid, query, answer, category)
            elif isinstance(val, dict):
                query = _first_key(val, _QUERY_KEYS)
                answer = _first_key(val, _ANSWER_KEYS)
                category = _first_key(val, _CATEGORY_KEYS)
                if query is not None:
                    add(vid, query, answer, category)
    else:
        raise ValueError(f"Unrecognized JSON top-level type: {type(data)}")

    return out

VIDEO_EXTS = [".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"]

def find_video_file(video_id, videos_dir):
    # exact filename match first (id already includes extension)
    direct = os.path.join(videos_dir, video_id)
    if os.path.isfile(direct):
        return direct
    # try id + known extensions
    for ext in VIDEO_EXTS:
        cand = os.path.join(videos_dir, f"{video_id}{ext}")
        if os.path.isfile(cand):
            return cand
    # fall back to glob in case of extra suffixes/case differences
    matches = glob.glob(os.path.join(videos_dir, f"{video_id}.*"))
    if matches:
        return matches[0]
    matches = glob.glob(os.path.join(videos_dir, f"{video_id}*"))
    if matches:
        return matches[0]
    return None

VIDEO_QUERIES = load_video_queries(JSON_PATH)
print(f"Loaded {sum(len(v) for v in VIDEO_QUERIES.values())} queries across {len(VIDEO_QUERIES)} video id(s)")
_n_with_category = sum(1 for items in VIDEO_QUERIES.values() for it in items if it.get("true_category"))
print(f"  {_n_with_category} of them already carry a ground-truth category from the JSON.")

# Resolve video files up front so missing videos are reported before any processing starts
RESOLVED = {}
MISSING = []
for vid in VIDEO_QUERIES:
    path = find_video_file(vid, VIDEOS_DIR)
    if path is None:
        MISSING.append(vid)
    else:
        RESOLVED[vid] = path

print(f"Resolved {len(RESOLVED)} video(s), missing {len(MISSING)}")
if MISSING:
    print("Missing video ids (no matching file in VIDEOS_DIR):")
    for vid in MISSING:
        print(f"  - {vid}")


## 4. API keys

In [ ]:
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (GPT-4o-mini): ")


## 5. Device + lazy model cache, with instrumentation

Models only load the first time a branch actually needs them, so unused branches never touch the GPU. `get_model` is wrapped so every load is logged with which query triggered it and how long it took, which is what lets section 17 report "models actually loaded" per question. GPU peak memory comes from `torch.cuda.max_memory_allocated` (exact, reset per call). CPU peak memory uses `resource.ru_maxrss`, a whole-process high-water mark, so we report its *growth* during a call -- a lower bound on that call's CPU memory, not an exact peak (it can under-count if RSS had already peaked earlier in the session).

In [ ]:
import torch, numpy as np
import time, resource
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

_MODEL_CACHE = {}
_ALL_MODEL_LOAD_EVENTS = []      # every model load: which query triggered it, how long it took
_CURRENT_MODEL_CONTEXT = set()   # models touched during the query/branch currently being timed
_CURRENT_QUERY_LABEL = None

def get_model(name, loader):
    if name not in _MODEL_CACHE:
        print(f"Loading {name} ...")
        t0 = time.perf_counter()
        _MODEL_CACHE[name] = loader()
        _ALL_MODEL_LOAD_EVENTS.append({
            "model": name,
            "load_time_s": round(time.perf_counter() - t0, 3),
            "triggered_by": _CURRENT_QUERY_LABEL,
        })
    _CURRENT_MODEL_CONTEXT.add(name)
    return _MODEL_CACHE[name]

def _cpu_rss_kb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss

def _reset_peak_memory():
    """Call before timing a block. Returns the CPU RSS baseline to diff against later."""
    if DEVICE == "cuda" and torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    return _cpu_rss_kb()

def _snapshot_peak_memory(cpu_baseline_kb):
    """Call after timing a block. Returns (peak_gpu_mb, cpu_rss_growth_mb)."""
    gpu_mb = (torch.cuda.max_memory_allocated() / 1024**2) if (DEVICE == "cuda" and torch.cuda.is_available()) else 0.0
    cpu_delta_mb = max(0, _cpu_rss_kb() - cpu_baseline_kb) / 1024
    return round(gpu_mb, 1), round(cpu_delta_mb, 1)

QUERY_METRICS = []      # one record per real (video, query) processed by the adaptive pipeline
COLD_RUN_METRICS = {}   # (video_id, category) -> cost of the ONE cold run of that branch on that video


## 6. Query classifier — GPT-4o-mini


In [ ]:
from openai import OpenAI
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

CLASSIFIER_MODEL = "gpt-4o-mini"
GENERATION_MODEL = "gpt-4o-mini"

CLASSIFY_PROMPT = """Classify the question about a video into exactly one category:
- temporal: ordering, sequence, before/after, when something happened
- spatial: layout, position, where something is at a single moment
- spatio_temporal: how something moved or changed position over time
- counting: how many of something
- audio: about speech, dialogue, or non-speech sounds

Reply with only the category name, nothing else.

Question: {query}
Category:"""

VALID_CATEGORIES = {"temporal", "spatial", "spatio_temporal", "counting", "audio"}
_CLASSIFIER_FALLBACK_COUNT = 0  # how often the model returned something outside VALID_CATEGORIES

def classify_query(query: str) -> str:
    resp = openai_client.chat.completions.create(
        model=CLASSIFIER_MODEL,
        messages=[{"role": "user", "content": CLASSIFY_PROMPT.format(query=query)}],
        temperature=0,
        max_tokens=200,
    )
    content = resp.choices[0].message.content
    label = content.strip().lower() if content else ""
    if label in VALID_CATEGORIES:
        return label
    global _CLASSIFIER_FALLBACK_COUNT
    _CLASSIFIER_FALLBACK_COUNT += 1  # counted in section 17c/17d as a distinct misroute cause
    return "spatial"  # safe fallback


## 6b. Canonical video id + human-labelled categories

`make_video_id()` is now the single place that turns a raw video id/filename into the sanitized id used everywhere (Chroma collection names, workdirs, and every metrics key below). It must be used consistently or the classification-accuracy join in section 17c silently comes up empty.

If the JSON already has a category field (`category`, `true_category`, `gt_category`, or `label`) it's picked up automatically. Otherwise you're prompted to hand-label each question once -- there is no way to score "classification accuracy against human-labelled categories" without an actual human providing those labels. Labels are cached to disk (`human_category_labels.json` in `WORKDIR`) so re-running this cell resumes instead of re-asking.

In [ ]:
import re

def make_video_id(raw_id: str) -> str:
    """Canonical sanitized video id, used everywhere a video needs a stable key."""
    raw_id = os.path.splitext(os.path.basename(str(raw_id)))[0]
    vid = re.sub(r"[^a-zA-Z0-9._-]+", "_", raw_id).strip("_-.")[:200]
    if len(vid) < 3:
        vid = f"video_{abs(hash(raw_id)) % 10**8}"
    return vid


In [ ]:
LABELS_PATH = os.path.join(WORKDIR, "human_category_labels.json")

def _load_saved_labels():
    if os.path.isfile(LABELS_PATH):
        with open(LABELS_PATH) as f:
            return json.load(f)
    return {}

_saved_labels = _load_saved_labels()  # key: "{video_id}||{query}" -> category

TRUE_CATEGORY = {}
_needs_labelling = []

for vid, items in VIDEO_QUERIES.items():
    video_id = make_video_id(vid)
    for item in items:
        key = f"{video_id}||{item['query']}"
        if item.get("true_category") in VALID_CATEGORIES:
            TRUE_CATEGORY[key] = item["true_category"]
        elif key in _saved_labels:
            TRUE_CATEGORY[key] = _saved_labels[key]
        else:
            _needs_labelling.append((key, vid, item["query"]))

print(f"{len(TRUE_CATEGORY)} question(s) already have a ground-truth category "
      f"(from the JSON or a previous labelling session).")
print(f"{len(_needs_labelling)} question(s) still need hand-labelling.")

CATS = sorted(VALID_CATEGORIES)  # deterministic order for the numeric shortcuts below

# Set to an int to hand-label only a random subset instead of every question
LABEL_SAMPLE_SIZE = None

to_label = _needs_labelling
if LABEL_SAMPLE_SIZE is not None and len(to_label) > LABEL_SAMPLE_SIZE:
    import random
    random.seed(0)
    to_label = random.sample(to_label, LABEL_SAMPLE_SIZE)

for key, vid, query in to_label:
    print(f"\nVideo: {vid}\nQuestion: {query}")
    for i, c in enumerate(CATS, 1):
        print(f"  {i}. {c}")
    while True:
        raw = input("Category [1-5, s=skip]: ").strip().lower()
        if raw == "s":
            break
        if raw.isdigit() and 1 <= int(raw) <= len(CATS):
            chosen = CATS[int(raw) - 1]
            TRUE_CATEGORY[key] = chosen
            _saved_labels[key] = chosen
            with open(LABELS_PATH, "w") as f:
                json.dump(_saved_labels, f, indent=2)
            break
        print("  not a valid choice, try again")

print(f"\n{len(TRUE_CATEGORY)}/{sum(len(v) for v in VIDEO_QUERIES.values())} "
      f"question(s) have a human category label.")


## 7. Shared ffmpeg utilities

In [ ]:
def extract_audio(video_path, out_wav):
    subprocess.run(
        ["ffmpeg", "-y", "-i", video_path, "-ar", "16000", "-ac", "1", out_wav],
        check=True, capture_output=True,
    )
    return out_wav

def extract_frames(video_path, out_dir, fps=1):
    os.makedirs(out_dir, exist_ok=True)
    subprocess.run(
        ["ffmpeg", "-y", "-i", video_path, "-vf", f"fps={fps}", f"{out_dir}/frame_%05d.jpg"],
        check=True, capture_output=True,
    )
    return sorted(glob.glob(f"{out_dir}/*.jpg"))

def caption_frame_gpt4o_mini(image_path, instruction="Describe this scene in one sentence."):
    import base64
    from openai import OpenAI
    client = get_model("openai_client", lambda: OpenAI())
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
            ],
        }],
        max_tokens=100,
    )
    return resp.choices[0].message.content.strip()


## 8. Temporal branch — PySceneDetect + GPT-4o-mini

In [ ]:
from scenedetect import detect, ContentDetector

def run_temporal_branch(video_path, query, workdir):
    scenes = detect(video_path, ContentDetector())
    frame_dir = os.path.join(workdir, "temporal_frames")
    os.makedirs(frame_dir, exist_ok=True)
    chunks = []
    for i, (start, end) in enumerate(scenes):
        mid_sec = (start.get_seconds() + end.get_seconds()) / 2
        frame_path = os.path.join(frame_dir, f"scene_{i:03d}.jpg")
        subprocess.run(
            ["ffmpeg", "-y", "-ss", str(mid_sec), "-i", video_path, "-frames:v", "1", frame_path],
            check=True, capture_output=True,
        )
        caption = caption_frame_gpt4o_mini(frame_path)
        chunks.append({
            "text": caption,
            "category": "temporal",
            "start_ts": start.get_seconds(),
            "end_ts": end.get_seconds(),
            "order": i,
        })
    return chunks


## 9. Spatial branch — CLIP keyframes + GPT-4o-mini + GroundingDINO

In [ ]:
from transformers import CLIPModel, CLIPProcessor
from sklearn.cluster import KMeans

def load_clip():
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    return model, processor

def select_keyframes(frame_paths, k=8):
    model, processor = get_model("clip", load_clip)
    embs = []
    for p in frame_paths:
        img = Image.open(p).convert("RGB")
        inputs = processor(images=img, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            emb = model.get_image_features(**inputs)
        # transformers >=5 returns a BaseModelOutputWithPooling instead of a plain tensor;
        # the projected image embedding lives in .pooler_output
        if hasattr(emb, "pooler_output"):
            emb = emb.pooler_output
        embs.append(emb.cpu().numpy().flatten())
    embs = np.stack(embs)
    k = min(k, len(frame_paths))
    labels = KMeans(n_clusters=k, n_init=5, random_state=0).fit_predict(embs)
    keyframes = []
    for c in range(k):
        idxs = np.where(labels == c)[0]
        center = embs[idxs].mean(axis=0)
        best = idxs[np.argmin(np.linalg.norm(embs[idxs] - center, axis=1))]
        keyframes.append(frame_paths[best])
    return sorted(keyframes)

def load_grounding_dino():
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
    processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
    model = AutoModelForZeroShotObjectDetection.from_pretrained("IDEA-Research/grounding-dino-tiny").to(DEVICE)
    return model, processor

def ground_objects(image_path, text_query):
    model, processor = get_model("grounding_dino", load_grounding_dino)
    img = Image.open(image_path).convert("RGB")
    inputs = processor(images=img, text=text_query, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    results = processor.post_process_grounded_object_detection(
    outputs, threshold=0.3,
    target_sizes=[img.size[::-1]],
    )[0]
    label_key = "text_labels" if "text_labels" in results else "labels"
    return [{"label": l, "box": b.tolist()} for l, b in zip(results[label_key], results["boxes"])]

def run_spatial_branch(video_path, query, workdir):
    frame_dir = os.path.join(workdir, "spatial_frames")
    frame_paths = extract_frames(video_path, frame_dir, fps=1)
    keyframes = select_keyframes(frame_paths, k=8)
    chunks = []
    for i, fp in enumerate(keyframes):
        caption = caption_frame_gpt4o_mini(fp, "Describe the layout and position of objects in this frame.")
        objects = ground_objects(fp, query)
        obj_str = ", ".join(o["label"] for o in objects) if objects else "none detected"
        chunks.append({
            "text": f"{caption} Objects grounded: {obj_str}.",
            "category": "spatial",
            "frame_id": i,
        })
    return chunks


## 10. Spatio-Temporal branch — RT-DETR + BoT-SORT

In [ ]:
import torch
from pathlib import Path

REID_MODEL_PATH = Path("osnet_x0_25_msmt17.pt")
REID_MODEL_URL = "https://github.com/mikel-brostrom/Yolov5_StrongSORT_OSNet/releases/download/v.1.2.7/osnet_x0_25_msmt17.pt"

def load_rtdetr():
    from transformers import RTDetrForObjectDetection, RTDetrImageProcessor
    processor = RTDetrImageProcessor.from_pretrained("PekingU/rtdetr_r50vd_coco_o365")
    model = RTDetrForObjectDetection.from_pretrained("PekingU/rtdetr_r50vd_coco_o365").to(DEVICE)
    return model, processor

def detect_objects_rtdetr(image_path, threshold=0.4):
    model, processor = get_model("rtdetr", load_rtdetr)
    img = Image.open(image_path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    results = processor.post_process_object_detection(
        outputs, target_sizes=torch.tensor([img.size[::-1]]), threshold=threshold,
    )[0]
    return np.column_stack([
        results["boxes"].cpu().numpy(),
        results["scores"].cpu().numpy(),
        results["labels"].cpu().numpy(),
    ])  # [x1, y1, x2, y2, score, class_id]

from scipy.optimize import linear_sum_assignment

def iou_batch(a, b):
    a, b = np.expand_dims(a, 1), np.expand_dims(b, 0)
    xx1 = np.maximum(a[..., 0], b[..., 0])
    yy1 = np.maximum(a[..., 1], b[..., 1])
    xx2 = np.minimum(a[..., 2], b[..., 2])
    yy2 = np.minimum(a[..., 3], b[..., 3])
    inter = np.maximum(0., xx2 - xx1) * np.maximum(0., yy2 - yy1)
    area_a = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1])
    area_b = (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    return inter / np.clip(area_a + area_b - inter, 1e-6, None)

class _Track:
    _next_id = 1
    def __init__(self, box, cls, conf):
        self.id = _Track._next_id; _Track._next_id += 1
        self.box = np.array(box, dtype=float)
        self.velocity = np.zeros(4)
        self.cls, self.conf = cls, conf
        self.hits, self.time_since_update = 1, 0

    def predict(self):
        self.box = self.box + self.velocity
        self.time_since_update += 1
        return self.box

    def update(self, box, conf):
        box = np.array(box, dtype=float)
        self.velocity = 0.5 * self.velocity + 0.5 * (box - self.box)
        self.box, self.conf = box, conf
        self.hits += 1
        self.time_since_update = 0

class SimpleSort:
    def __init__(self, iou_threshold=0.3, max_age=5, min_hits=1):
        self.iou_threshold, self.max_age, self.min_hits = iou_threshold, max_age, min_hits
        self.tracks = []

    def update(self, dets, img=None):
        dets = np.asarray(dets, dtype=float) if len(dets) else np.empty((0, 6))
        for t in self.tracks:
            t.predict()

        if not self.tracks:
            matches, unmatched_dets, unmatched_tracks = [], list(range(len(dets))), []
        elif len(dets) == 0:
            matches, unmatched_dets, unmatched_tracks = [], [], list(range(len(self.tracks)))
        else:
            track_boxes = np.array([t.box for t in self.tracks])
            iou_mat = iou_batch(dets[:, :4], track_boxes)
            for i, d in enumerate(dets):
                for j, t in enumerate(self.tracks):
                    if d[5] != t.cls:
                        iou_mat[i, j] = 0
            row_ind, col_ind = linear_sum_assignment(-iou_mat)
            matches, unmatched_dets, unmatched_tracks = [], list(range(len(dets))), list(range(len(self.tracks)))
            for r, c in zip(row_ind, col_ind):
                if iou_mat[r, c] >= self.iou_threshold:
                    matches.append((r, c))
                    unmatched_dets.remove(r)
                    unmatched_tracks.remove(c)

        for d_idx, t_idx in matches:
            self.tracks[t_idx].update(dets[d_idx, :4], dets[d_idx, 4])
        for d_idx in unmatched_dets:
            self.tracks.append(_Track(dets[d_idx, :4], dets[d_idx, 5], dets[d_idx, 4]))
        self.tracks = [t for t in self.tracks if t.time_since_update <= self.max_age]

        rows = [
            [*t.box, t.id, t.cls, t.conf]
            for t in self.tracks if t.time_since_update == 0 and t.hits >= self.min_hits
        ]
        return np.array(rows) if rows else np.empty((0, 7))

def load_tracker():
    return SimpleSort(iou_threshold=0.3, max_age=5, min_hits=1)
def run_spatiotemporal_branch(video_path, query, workdir, fps=2):
    frame_dir = os.path.join(workdir, "st_frames")
    frame_paths = extract_frames(video_path, frame_dir, fps=fps)
    tracker = get_model("bot_sort_st", load_tracker)
    tracks = {}
    for idx, fp in enumerate(frame_paths):
        dets = detect_objects_rtdetr(fp)
        img = np.array(Image.open(fp).convert("RGB"))
        tracked = tracker.update(dets, img) if len(dets) else []
        for row in tracked:
            x1, y1, x2, y2, tid, cls, conf = row[:7]
            tracks.setdefault(int(tid), []).append((idx, (x1, y1, x2, y2), int(cls)))
    chunks = []
    for tid, path in tracks.items():
        start_idx, start_box, cls = path[0]
        end_idx, end_box, _ = path[-1]
        text = (f"Object class {cls} (track {tid}) moved from {region_of(start_box)} to "
                f"{region_of(end_box)} between t={start_idx/fps:.1f}s and t={end_idx/fps:.1f}s.")
        chunks.append(
            {"text": text, "category": "spatio_temporal",
            "track_id": tid, "start_ts": start_idx / fps, "end_ts": end_idx / fps,
        })
    return chunks

## 11. Counting branch — RT-DETR + BoT-SORT dedup

In [ ]:
def run_counting_branch(video_path, query, workdir, fps=1):
    frame_dir = os.path.join(workdir, "count_frames")
    frame_paths = extract_frames(video_path, frame_dir, fps=fps)
    tracker = get_model("bot_sort_count", load_tracker)
    class_track_ids = {}
    evidence = {}
    for fp in frame_paths:
        dets = detect_objects_rtdetr(fp)
        img = np.array(Image.open(fp).convert("RGB"))
        tracked = tracker.update(dets, img) if len(dets) else []
        for row in tracked:
            x1, y1, x2, y2, tid, cls, conf = row[:7]
            class_track_ids.setdefault(int(cls), set()).add(int(tid))
            evidence.setdefault(int(cls), fp)
    chunks = []
    for cls, ids in class_track_ids.items():
        chunks.append({
            "text": f"{len(ids)} unique instance(s) of class {cls} observed across the video.",
            "category": "counting",
            "class_id": cls, "count": len(ids), "evidence_frame": evidence[cls],
        })
    return chunks


## 12. Audio branch — Whisper large-v3-turbo + AST

In [ ]:
from transformers import pipeline as hf_pipeline

def load_whisper():
    return hf_pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-large-v3-turbo",
        device=0 if DEVICE == "cuda" else -1,
        chunk_length_s=30,
        return_timestamps=True,
    )

def load_ast():
    return hf_pipeline(
        "audio-classification",
        model="MIT/ast-finetuned-audioset-10-10-0.4593",
        device=0 if DEVICE == "cuda" else -1,
    )

def run_audio_branch(video_path, query, workdir):
    wav_path = os.path.join(workdir, "audio.wav")
    extract_audio(video_path, wav_path)

    asr = get_model("whisper", load_whisper)
    transcript = asr(wav_path)
    segments = transcript.get("chunks") or [{"text": transcript["text"], "timestamp": (0, None)}]

    ast = get_model("ast", load_ast)
    tags = ast(wav_path, top_k=5)
    tag_str = ", ".join(t["label"] for t in tags)

    chunks = []
    for seg in segments:
        start, end = seg["timestamp"]
        text = seg["text"].strip()
        if not text:
            continue
        chunks.append({
            "text": text, "category": "audio",
            "start_ts": start, "end_ts": end, "event_tags": tag_str,
        })
    return chunks


## 13. Context assembly — BGE embeddings in ChromaDB, hybrid retrieval

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from rank_bm25 import BM25Okapi

def load_bge():
    return SentenceTransformer("BAAI/bge-large-en-v1.5", device=DEVICE)

chroma_client = chromadb.PersistentClient(path=os.path.join(WORKDIR, "chroma"))

def get_collection(video_id):
    return chroma_client.get_or_create_collection(name=f"video_{video_id}")

def _sanitize_metadata(md):
    # Chroma metadata values must be str/int/float/bool -- no None, list, or dict.
    # e.g. Whisper leaves end_ts=None when it cannot predict a segment's ending timestamp.
    clean = {}
    for k, v in md.items():
        if v is None:
            continue  # drop rather than coerce, so missing timestamps don't become the string "None"
        if isinstance(v, (str, int, float, bool)):
            clean[k] = v
        else:
            clean[k] = str(v)
    return clean

def add_chunks(video_id, chunks):
    if not chunks:
        return
    bge = get_model("bge", load_bge)
    coll = get_collection(video_id)
    texts = [c["text"] for c in chunks]
    embeddings = bge.encode(texts, normalize_embeddings=True).tolist()
    ids = [f"{video_id}_{c['category']}_{i}" for i, c in enumerate(chunks)]
    metadatas = [_sanitize_metadata({k: v for k, v in c.items() if k != "text"}) for c in chunks]
    coll.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)

def hybrid_retrieve(video_id, query, category, top_k=6):
    bge = get_model("bge", load_bge)
    coll = get_collection(video_id)
    q_emb = bge.encode([query], normalize_embeddings=True).tolist()
    dense = coll.query(query_embeddings=q_emb, n_results=top_k, where={"category": category})
    docs, metas = dense["documents"][0], dense["metadatas"][0]

    if len(docs) > 1:
        tokenized = [d.lower().split() for d in docs]
        bm25 = BM25Okapi(tokenized)
        scores = bm25.get_scores(query.lower().split())
        order = np.argsort(scores)[::-1]
        docs = [docs[i] for i in order]
        metas = [metas[i] for i in order]

    return list(zip(docs, metas))


## 14. Answer generation — GPT-4o-mini, category-specific prompts


In [ ]:
PROMPT_TEMPLATES = {
    "temporal": "Use the timestamped scene descriptions below, in order, to answer the question about sequence or timing.",
    "spatial": "Use the frame descriptions below, which describe layout and object positions, to answer the question.",
    "spatio_temporal": "Use the object trajectory descriptions below to answer the question about movement over time.",
    "counting": "Use the count records below. Report the exact number(s) given, do not estimate.",
    "audio": "Use the transcript and audio event tags below to answer the question about speech or sound.",
}

def generate_answer(query, category, retrieved):
    context = "\n".join(f"- {doc}" for doc, _ in retrieved) or "(no context retrieved)"
    instruction = PROMPT_TEMPLATES.get(category, "Use the context below to answer the question.")
    prompt = f"{instruction}\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    resp = openai_client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=1024,
    )
    content = resp.choices[0].message.content
    return content.strip() if content else "(model returned no content — try increasing max_tokens)"


## 15. Orchestration

Only the classified branch's function is ever called -- the other four never run. `process_query` now also times every stage (classify / branch / retrieve / generate), snapshots peak GPU + CPU memory, and records exactly which models were touched, all logged to `QUERY_METRICS` and `COLD_RUN_METRICS` for the evaluation in section 17.

In [ ]:
import re
import traceback

BRANCH_FUNCS = {
    "temporal": run_temporal_branch,
    "spatial": run_spatial_branch,
    "spatio_temporal": run_spatiotemporal_branch,
    "counting": run_counting_branch,
    "audio": run_audio_branch,
}

def process_query(video_path, query, video_id_hint=None):
    global _CURRENT_QUERY_LABEL
    try:
        video_id = video_id_hint or make_video_id(video_path)
        _CURRENT_QUERY_LABEL = f"{video_id}: {query[:60]}"
        _CURRENT_MODEL_CONTEXT.clear()
        cpu_base = _reset_peak_memory()
        t_start = time.perf_counter()

        category = classify_query(query)
        t_classified = time.perf_counter()
        print(f"[{video_id}] '{query}' -> {category}")

        video_workdir = os.path.join(WORKDIR, video_id)
        os.makedirs(video_workdir, exist_ok=True)

        coll = get_collection(video_id)
        existing = coll.get(where={"category": category}) if coll.count() else {"ids": []}
        cache_hit = bool(existing["ids"])

        t_branch_start = time.perf_counter()
        if not cache_hit:
            print(f"  running {category} branch (no cached chunks yet)")
            chunks = BRANCH_FUNCS[category](video_path, query, video_workdir)
            print(f"  {category} branch produced {len(chunks)} chunk(s)")
            add_chunks(video_id, chunks)
        else:
            print(f"  reusing {len(existing['ids'])} cached {category} chunk(s) for {video_id}")
        t_branch_end = time.perf_counter()

        retrieved = hybrid_retrieve(video_id, query, category)
        print(f"  retrieved {len(retrieved)} chunk(s) for generation")
        t_retrieved = time.perf_counter()
        answer = generate_answer(query, category, retrieved)
        t_end = time.perf_counter()

        gpu_mb, cpu_mb = _snapshot_peak_memory(cpu_base)
        branch_time = (t_branch_end - t_branch_start) if not cache_hit else 0.0

        metrics = {
            "video": video_id, "query": query, "category": category,
            "cache_hit": cache_hit,
            "t_classify_s": round(t_classified - t_start, 3),
            "t_branch_s": round(branch_time, 3),
            "t_retrieve_s": round(t_retrieved - t_branch_end, 3),
            "t_generate_s": round(t_end - t_retrieved, 3),
            "t_total_s": round(t_end - t_start, 3),
            "peak_gpu_mb": gpu_mb, "peak_cpu_delta_mb": cpu_mb,
            "models_loaded_this_call": sorted(_CURRENT_MODEL_CONTEXT),
            "n_models_loaded_this_call": len(_CURRENT_MODEL_CONTEXT),
        }
        QUERY_METRICS.append(metrics)
        if not cache_hit:
            # Cost of the ONE cold run of this branch on this video -- reused by the section-17
            # baseline runner instead of re-paying for the same branch a second time.
            COLD_RUN_METRICS[(video_id, category)] = {
                "time_s": round(branch_time, 3), "peak_gpu_mb": gpu_mb, "peak_cpu_delta_mb": cpu_mb,
                "models_loaded": sorted(_CURRENT_MODEL_CONTEXT),
            }

        return {
            "video": video_id, "query": query, "category": category, "answer": answer,
            "cache_hit": cache_hit, "latency_s": metrics["t_total_s"],
            "peak_gpu_mb": gpu_mb, "peak_cpu_delta_mb": cpu_mb,
            "models_loaded": metrics["models_loaded_this_call"],
        }
    except Exception:
        print(f"--- process_query failed on: {query!r} ---")
        traceback.print_exc()
        raise
    finally:
        _CURRENT_QUERY_LABEL = None


In [ ]:
_MODEL_CACHE.pop("bot_sort_st", None)
_MODEL_CACHE.pop("bot_sort_count", None)
print("cleared stale tracker cache")

## 16. Run every (video, query) pair from the JSON and save results

Videos are processed in order of first appearance; each video's branch chunks are cached in
ChromaDB (see cell 31 `process_query`), so repeat questions about the same video reuse cached
chunks instead of re-running detection/ASR/etc.

In [ ]:
import subprocess
import glob
import json
results = []
total = sum(len(v) for v in VIDEO_QUERIES.values())
done = 0

for vid, items in VIDEO_QUERIES.items():
    video_path = RESOLVED.get(vid)
    if video_path is None:
        for item in items:
            done += 1
            print(f"[{done}/{total}] SKIP missing video '{vid}': {item['query']!r}")
            results.append({
                "video": vid,
                "query": item["query"],
                "expected_answer": item.get("expected_answer"),
                "error": "video file not found in VIDEOS_DIR",
            })
        continue

    for item in items:
        done += 1
        query = item["query"]
        print(f"[{done}/{total}] video={vid!r} query={query!r}")
        try:
            result = process_query(video_path, query)
            if result is None:
                result = {"video": vid, "query": query, "error": "process_query returned no result"}
            result["expected_answer"] = item.get("expected_answer")
            results.append(result)
        except Exception as e:
            print(f"  Error on '{query}' for video '{vid}': {e}")
            results.append({
                "video": vid,
                "query": query,
                "expected_answer": item.get("expected_answer"),
                "error": str(e),
            })



for r in results:
    if not r:
        print("(empty result, skipped)\n")
        continue
    print(f"[{r.get('video')}] Q: {r.get('query')}")
    print(f"  [{r.get('category', 'error')}] {r.get('answer', r.get('error'))}")
    if r.get('expected_answer') is not None:
        print(f"  expected: {r.get('expected_answer')}")
    print()


## 16b. Baseline: cost of running every branch (no adaptive routing)

The framework's only real novelty is skipping branches -- so that saving has to be measured against a system that doesn't skip them. For every video, each of the 5 branches is run at least once and its cost is captured. Where a branch's cold-run cost was already measured during the real adaptive run above (`COLD_RUN_METRICS`), that measurement is reused instead of paying for the same branch twice. Anything not yet run is executed now into an isolated `_baseline_only` workdir and a separate model-cache context, so it never touches the real per-video ChromaDB cache used to answer the actual questions -- this is a timing/memory-only run, its output is discarded.

This does mean real GPU/API time is spent running branches nobody asked for -- that's the point: it's the honest cost of the always-run-all-branches system this pipeline is compared against.

In [ ]:
BASELINE_METRICS = {}  # video_id -> {category: {time_s, peak_gpu_mb, peak_cpu_delta_mb, models_loaded, source}}

def run_baseline_all_branches():
    global _CURRENT_QUERY_LABEL
    for vid, video_path in RESOLVED.items():
        video_id = make_video_id(vid)
        video_workdir = os.path.join(WORKDIR, video_id)
        BASELINE_METRICS.setdefault(video_id, {})
        for category, fn in BRANCH_FUNCS.items():
            key = (video_id, category)
            if key in COLD_RUN_METRICS:
                m = COLD_RUN_METRICS[key]
                BASELINE_METRICS[video_id][category] = {**m, "source": "reused_real_run"}
                continue

            baseline_dir = os.path.join(video_workdir, "_baseline_only", category)
            os.makedirs(baseline_dir, exist_ok=True)
            _CURRENT_QUERY_LABEL = f"[baseline] {video_id}: {category}"
            _CURRENT_MODEL_CONTEXT.clear()
            cpu_base = _reset_peak_memory()
            t0 = time.perf_counter()
            status = "ok"
            try:
                fn(video_path, "(baseline timing run, output discarded)", baseline_dir)
            except Exception as e:
                status = f"error: {type(e).__name__}: {e}"
            elapsed = time.perf_counter() - t0
            gpu_mb, cpu_mb = _snapshot_peak_memory(cpu_base)
            BASELINE_METRICS[video_id][category] = {
                "time_s": round(elapsed, 3), "peak_gpu_mb": gpu_mb, "peak_cpu_delta_mb": cpu_mb,
                "models_loaded": sorted(_CURRENT_MODEL_CONTEXT), "source": "baseline_only_run",
                "status": status,
            }
            _CURRENT_QUERY_LABEL = None
    return BASELINE_METRICS

# Set False to skip -- this executes every branch at least once per video and costs real GPU/API time
RUN_BASELINE = True
if RUN_BASELINE:
    run_baseline_all_branches()
    print(f"Baseline measured for {len(BASELINE_METRICS)} video(s).")
else:
    print("Baseline skipped (RUN_BASELINE = False); savings numbers below will be unavailable.")


## 17. Evaluation: latency, memory, and compute saved

Per-question numbers come straight from `QUERY_METRICS` (the real adaptive run). Per-video savings compare the branches actually run for that video (`COLD_RUN_METRICS`) against every branch running (`BASELINE_METRICS`).

In [ ]:
import pandas as pd

per_query_df = pd.DataFrame(QUERY_METRICS)
print(f"Logged metrics for {len(per_query_df)} question(s).\n")
if len(per_query_df):
    display_cols = ["video", "category", "cache_hit", "t_total_s", "t_branch_s",
                     "peak_gpu_mb", "peak_cpu_delta_mb", "n_models_loaded_this_call"]
    print("Per-question metrics:")
    print(per_query_df[display_cols].to_string(index=False))


In [ ]:
# ---- per-video: adaptive cost actually paid vs. baseline cost of running every branch ----
rows = []
for video_id, cats in BASELINE_METRICS.items():
    baseline_time = sum(c["time_s"] for c in cats.values())
    baseline_models = sorted(set(m for c in cats.values() for m in c["models_loaded"]))
    categories_run = {cat for (vid, cat) in COLD_RUN_METRICS if vid == video_id}
    adaptive_time = sum(COLD_RUN_METRICS[(video_id, cat)]["time_s"] for cat in categories_run)
    adaptive_models = sorted(set(
        m for cat in categories_run for m in COLD_RUN_METRICS[(video_id, cat)]["models_loaded"]
    ))
    rows.append({
        "video": video_id,
        "branches_run": len(categories_run),
        "branches_total": len(BRANCH_FUNCS),
        "adaptive_branch_time_s": round(adaptive_time, 2),
        "baseline_all_branch_time_s": round(baseline_time, 2),
        "time_saved_s": round(baseline_time - adaptive_time, 2),
        "time_saved_pct": round(100 * (baseline_time - adaptive_time) / baseline_time, 1) if baseline_time else float("nan"),
        "models_loaded_adaptive": len(adaptive_models),
        "models_loaded_baseline": len(baseline_models),
    })
per_video_savings_df = pd.DataFrame(rows)
print("Per-video compute saved (branch execution time only, GPU/API calls):")
print(per_video_savings_df.to_string(index=False))

if len(per_video_savings_df):
    tot_adaptive = per_video_savings_df["adaptive_branch_time_s"].sum()
    tot_baseline = per_video_savings_df["baseline_all_branch_time_s"].sum()
    if tot_baseline:
        print(f"\nTotal branch compute time: adaptive={tot_adaptive:.1f}s vs. "
              f"run-all-branches={tot_baseline:.1f}s "
              f"({100*(tot_baseline-tot_adaptive)/tot_baseline:.1f}% saved)")


In [ ]:
# ---- per-category latency / memory summary from the real adaptive run ----
if len(per_query_df):
    per_category_df = per_query_df.groupby("category").agg(
        n_questions=("query", "count"),
        mean_total_latency_s=("t_total_s", "mean"),
        mean_branch_latency_s=("t_branch_s", "mean"),
        mean_peak_gpu_mb=("peak_gpu_mb", "mean"),
        mean_peak_cpu_delta_mb=("peak_cpu_delta_mb", "mean"),
        mean_models_loaded=("n_models_loaded_this_call", "mean"),
        cache_hit_rate=("cache_hit", "mean"),
    ).round(2)
    print("Per-category summary (adaptive run):")
    print(per_category_df.to_string())
else:
    per_category_df = pd.DataFrame()


## 17c. Classification accuracy & confusion matrix

The whole pipeline depends on this first step routing correctly -- an audio question sent to the spatial branch means Whisper never runs and the correct answer becomes unreachable regardless of anything downstream. This scores `classify_query`'s output against the human-labelled categories from section 6b.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score

eval_rows = []
for m in QUERY_METRICS:
    key = f"{m['video']}||{m['query']}"
    true_cat = TRUE_CATEGORY.get(key)
    if true_cat is None:
        continue
    eval_rows.append({
        "video": m["video"], "query": m["query"],
        "predicted": m["category"], "true": true_cat,
        "misrouted": m["category"] != true_cat,
    })
eval_df = pd.DataFrame(eval_rows)

if len(eval_df) == 0:
    print("No human-labelled categories available -- run the labelling cell in section 6b first.")
    cm_df = pd.DataFrame()
else:
    labels = sorted(VALID_CATEGORIES)
    acc = accuracy_score(eval_df["true"], eval_df["predicted"])
    cm = confusion_matrix(eval_df["true"], eval_df["predicted"], labels=labels)
    cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
    print(f"Classification accuracy vs. {len(eval_df)} human-labelled question(s): {acc:.1%}\n")
    print("Confusion matrix (rows = true category, columns = predicted category):")
    print(cm_df.to_string())
    n_misrouted = int(eval_df["misrouted"].sum())
    print(f"\n{n_misrouted}/{len(eval_df)} question(s) ({n_misrouted/len(eval_df):.1%}) were misrouted.")
    if _CLASSIFIER_FALLBACK_COUNT:
        print(f"({_CLASSIFIER_FALLBACK_COUNT} of ALL classifier calls this session fell back to the "
              f"default 'spatial' label because the model returned something outside the 5 categories.)")


## 17d. Does a misroute actually break the answer?

A misroute is only catastrophic if it produces a wrong final answer -- e.g. an audio question sent to spatial might still coincidentally retrieve something usable. This uses an LLM judge (same GPT-4o-mini model as the classifier) to compare each generated answer against the `expected_answer` from the JSON, then splits accuracy by whether the question was misrouted.


In [ ]:
JUDGE_PROMPT = """You are grading a video-QA system's answer against a reference answer.
Question: {query}
Reference answer: {expected}
System answer: {answer}

Does the system answer convey the same information as the reference answer? Minor wording
differences are fine. Reply with only one word: CORRECT or INCORRECT."""

def judge_answer(query, expected, answer):
    if not expected:
        return None
    resp = openai_client.chat.completions.create(
        model=CLASSIFIER_MODEL,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(query=query, expected=expected, answer=answer)}],
        temperature=0, max_tokens=10,
    )
    content = (resp.choices[0].message.content or "").strip().upper()
    return "CORRECT" in content

# expected_answer + generated answer live in `results` from the main loop (section 16)
answer_lookup = {(r["video"], r["query"]): r for r in results if r.get("video")}

judged_rows = []
if len(eval_df):
    for _, row in eval_df.iterrows():
        r = answer_lookup.get((row["video"], row["query"]))
        if r is None or r.get("error"):
            continue
        expected = r.get("expected_answer")
        if not expected:
            continue
        correct = judge_answer(row["query"], expected, r.get("answer", ""))
        if correct is None:
            continue
        judged_rows.append({**row.to_dict(), "answer_correct": correct})

judged_df = pd.DataFrame(judged_rows)

if len(judged_df) == 0:
    print("No question had both a human category label and an expected_answer to judge against.")
else:
    has_misroutes = judged_df["misrouted"].any()
    routed_acc = judged_df.loc[~judged_df["misrouted"], "answer_correct"].mean()
    n_routed = int((~judged_df["misrouted"]).sum())
    print(f"Answer accuracy when correctly routed: {routed_acc:.1%}  (n={n_routed})")
    if has_misroutes:
        misrouted_acc = judged_df.loc[judged_df["misrouted"], "answer_correct"].mean()
        n_mis = int(judged_df["misrouted"].sum())
        print(f"Answer accuracy when misrouted:         {misrouted_acc:.1%}  (n={n_mis})")
    else:
        print("Answer accuracy when misrouted:         n/a (no misroutes in the labelled sample)")

    n_wrong = int((~judged_df["answer_correct"]).sum())
    n_wrong_and_misrouted = int(((~judged_df["answer_correct"]) & judged_df["misrouted"]).sum())
    if n_wrong:
        print(f"\n{n_wrong} wrong answer(s) total; {n_wrong_and_misrouted}/{n_wrong} "
              f"({n_wrong_and_misrouted/n_wrong:.1%}) of them followed a misroute.")
    else:
        print("\nNo wrong answers in the judged sample.")


## 17e. Save metrics alongside results

Writes the per-question metrics, per-video savings, confusion matrix, and misroute-impact tables as CSVs next to `OUTPUT_JSON`, so the numbers above survive the Colab session.

In [ ]:
METRICS_DIR = os.path.dirname(OUTPUT_JSON) or "."
per_query_path = os.path.join(METRICS_DIR, "metrics_per_query.csv")
per_video_path = os.path.join(METRICS_DIR, "metrics_per_video_savings.csv")
confusion_path = os.path.join(METRICS_DIR, "metrics_confusion_matrix.csv")
judged_path = os.path.join(METRICS_DIR, "metrics_misroute_impact.csv")

per_query_df.to_csv(per_query_path, index=False)
if "per_video_savings_df" in globals() and len(per_video_savings_df):
    per_video_savings_df.to_csv(per_video_path, index=False)
if "cm_df" in globals() and len(cm_df):
    cm_df.to_csv(confusion_path)
if "judged_df" in globals() and len(judged_df):
    judged_df.to_csv(judged_path, index=False)

print("Saved:")
for p in [per_query_path, per_video_path, confusion_path, judged_path]:
    if os.path.isfile(p):
        print(" -", p)


## 18. (Optional) also download the results file locally

Results are already saved to OUTPUT_JSON on Drive (see above). This just also pops a local
download if you want a copy outside Drive.

In [ ]:
from google.colab import files as colab_files
colab_files.download(OUTPUT_JSON)
